# Training eines neuronalen Netzes zur Erkennung von handschriftlichen Ziffern

## 0) Ziel
Es soll ein KI-System entwickelt werden, das mithilfe eines neuronalen Netzes handschriftliche Ziffern erkennt. Das System könnte z. B. bei der automatisierten Erkennung und Digitalisierung von Ziffern in folgenden Anwendungsfällen eingesetzt werden:
* Note auf einer Schülerarbeit oder von handschriftlichen Notenbögen
* Postleitzahlen auf Briefen
* handschriftliche Steuerformulare oder Überweisungsträger
* Handschrifterkennung auf einem Tablet

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 1</b></p>
<ol>
   <li style="list-style-type: lower-alpha;">Diskutieren Sie (insgesamt maximal 10 Minuten) für zwei Anwendungsfälle die Chancen und Risiken des Einsatzes eines solchen KI-Systems. Erläutern Sie dabei insbesondere die Auswirkungen für den Fall, dass eine bestimmte Ziffer falsch erkannt wird.</li>
    <li style="list-style-type: lower-alpha;">Ein KI-System wird nie zu 100% korrekt klassifizieren können. Geben Sie für alle gegebenen Anwendungsfälle einen intuitiven Wert für die Genauigkeit an, den Sie erwarten würden, um das System einzusetzen. Notieren Sie sich die Werte oben hinter den Anwendungsfällen.</li>
</ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>

# Übersicht
Dieses Jupyter Notebook erarbeitet schrittweise die Erstellung eines neuronalen Netzes. Dazu werden folgende Schritte durchlaufen


1.   Importieren der notwendigen Bibliotheken
2.   Laden der Trainings- und Testdaten
3.   Analyse der Daten
4.   Vorbereitung der Daten fürs Training
5.   Neuronales Netz definieren
6.   Neuronales Netz trainieren
7.   Netz mit Testdaten testen
8.   Netz auf neue Daten anwenden
9.   Ausblick auf komplexe Netze





## 1) Importieren der notwendigen Bibliotheken
Das eigentliche neuronale Netz wird mit keras erstellt. Keras ist ein Teil der Bibliothek tensorflow. Numpy und matplotlib dienen für mathematische Berechnungen sowie zur Visualisierung. PIL wird ganz am Ende der Einheit verwendet, um ein Bild im jpeg-Format so umzuwandeln, dass es vom neuronalen Netz klassifiziert werden kann.
<br><b>Hinweis:</b> Beim Ausführen des Imports werden unter Umständen rote Warnungen angezeigt. Diese können Sie ignorieren.


In [ ]:
#Warnungen unterdrücken
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import tensorflow as tf

import matplotlib.pyplot as plt
from PIL import Image

## 2) Daten laden

Der MNIST-Datensatz wird geladen. Er besteht aus handgeschriebenen Ziffern. Der Datensatz enthält zwei Tupel. Eines für die Trainingsdaten und eines für die Testdaten. Jedes dieser Tupel enthält zwei Elemente:
* Die eigentlichen Bilddaten: Für jedes Bild werden die Grauwerte der Pixel in einer Matrix (Array von Arrays) gespeichert.
* Ein Array, das die Labels zu den Bildern enthält.

In [ ]:
(train_data, train_labels), (test_data, test_labels) = tf.keras.datasets.mnist.load_data()

## 3) Analyse des Datensatzes
Da das Ergebnis eines neuronalen Netzes von den Trainingsdaten abhängt, ist es wichtig, sich zunächst mit dem Datensatz vertraut zu machen. Dazu wird dieser im folgenden Abschnitt anhand von Beispielen visualisiert und anschließend hinsichtlich Aspekte wie der Länge von Trainings- und Testdaten sowie der Verteilung der unterschiedlichen Ziffern analysiert.

Zunächst werden die Grauwerte einer Ziffer aus dem Trainingsdatensatz angezeigt. Anschließend wird diese Ziffer als Bild dargestellt.

In [ ]:
print(f"Das Label des Beispielbildes ist: {train_labels[0]}")
#Grauwerte als Matrix ausgeben
for row in train_data[0]:
    print(" ".join(f"{pixel:3d}" for pixel in row))  # Formatierung der Ausgabe mit passenden Zeilenumbrüchen

In [ ]:
# Bild anzeigen
plt.imshow(train_data[0], cmap='gray')  # Bild im Graustufenmodus anzeigen
plt.title(f"Label: {train_labels[0]}")  # Titel mit Label des Bildes
plt.show()

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 2</b></p>
<ol>
   <li style="list-style-type: lower-alpha;">Die Anzahl der Neuronen in der Eingabeschicht des neuronalen Netzes soll der Anzahl der Pixel eines Bildes entsprechen. Ermitteln Sie daher (z. B. anhand der Visualisierung am Anfang des Abschnitts) die Anzahl der Pixel eines Bildes?</li>
    <li style="list-style-type: lower-alpha;">Lassen Sie sich andere Ziffern aus dem Trainingsdatensatz anzeigen. Verändern Sie dazu entweder den obigen Quelltext oder fügen Sie unterhalb dieses Arbeitsauftrags eine neue Zelle hinzu.</li>
</ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>

Um ein Gefühl für die Darstellung der Ziffer 1 zu bekommen, werden die ersten 21 Vorkommen der Ziffer 1 visualisiert.

In [ ]:
# Indizes der Bilder mit der Ziffer 1 finden
indices_of_ones = np.where(train_labels == 1)[0]

# Visualisiere die ersten 21 Vorkommen der Ziffer 1
plt.figure(figsize=(21, 7))
for i in range(21):
    plt.subplot(3, 7, i + 1)  # Erstelle 3 Reihen mit 7 Bildern
    plt.imshow(train_data[indices_of_ones[i]], cmap='gray')
    plt.title(f"Label: 1")
    plt.axis('off')  # Keine Achsen anzeigen
plt.tight_layout()  # Sorgt dafür, dass die Bilder ordentlich angeordnet sind
plt.show()

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 3</b></p>
<ol>
   <li style="list-style-type: lower-alpha;">Decken sich die Bilder der Ziffer 1 mit Ihren Erwartungen? Wie zeichnen Sie die 1?</li>
     <li style="list-style-type: lower-alpha;">Erläutern Sie, welche Probleme unterschiedliche Zeichnungen einer Ziffer für das Training sowie den Einsatz des Systems mit sich bringen können.</li>
    <li style="list-style-type: lower-alpha;">Lassen Sie sich eine andere Ziffer anzeigen und überprüfen Sie die Ausgabe mit Ihren Erwartungen. Verändern Sie dazu entweder den Quelltext oben oder fügen Sie unterhalb dieses Arbeitsauftrages eine neue Zelle ein.</li>
</ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>

Um ein Gefühl für die Darstellung aller Ziffern zu bekommen, wird im Folgenden eine durchschnittliche Darstellung jeder Ziffer angezeigt.

In [ ]:
avg_images = []
for digit in range(10):
    digit_images = train_data[train_labels == digit]
    avg_image = np.mean(digit_images, axis=0)  # Durchschnittliches Bild berechnen
    avg_images.append(avg_image)

# Visualisierung der Durchschnittsbilder
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(avg_images[i], cmap='gray')
    plt.title(f"durchschnittliche {i}")
    plt.axis('off')
plt.tight_layout()
plt.show()

Durch die Visualisierungen haben Sie einen intuitiven Einblick in die Trainingsdaten erhalten. Dieser soll jetzt durch eine statistische Analyse einfacher Aspekte erweitert werden.

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 4</b></p>

Finden Sie heraus, wie viele Trainingsdaten und wie viele Testdaten im Datensatz enthalten sind und berechnen Sie jeweils den prozentualen Anteil am gesamten Datensatz. Verwenden Sie die Zelle unterhalb des Arbeitsauftrages.
<br>(Hinweis 1: Verwenden Sie die Python-Funktion len(s))<br>
<br>
<details>
<summary>Klicken für Hinweis 2</summary>
<p>Die Trainings- und Testdaten können der Funktion len(s) als Parameter übergeben werden.</p>
</details>
<p style="background-color: yellow; text-align:center">&nbsp</p>

In [ ]:
# Arbeitsauftrag: Länge von Trainingsdaten und Testdaten sowie Verhältnis


Es wird überprüft, ob die Daten tatsächlich 10 unterschiedliche Ziffern enthalten.

In [ ]:
unique_labels = np.unique(train_labels)
print(f"Folgende Label werden in den Trainingsdaten vergeben: {unique_labels}")

Wie ist die Verteilung der Ziffern in den Trainingsdaten?

In [ ]:
digit_counts = np.bincount(train_labels)
print(f"Häufigkeit der Ziffern im Trainingsdatensatz:")
for digit, count in enumerate(digit_counts):
    print(f"Ziffer {digit}: {count} Vorkommen")

Diese Häufigkeit kann man auch anschaulich visualisieren:

In [ ]:
plt.bar(range(10), digit_counts)
plt.xlabel('Ziffer')
plt.ylabel('Anzahl der Vorkommen')
plt.title('Verteilung der Ziffern im Trainingsdatensatz')
plt.xticks(range(10))
plt.show()

<p style= "background-color: yellow;text-align:center"><b>ARBEITSAUFTRAG 5</b>
</p>

Überprüfen Sie, ob die einzelnen Ziffern in den Testdaten genauso häufig vorkommen wie in den Trainingsdaten. Ergänzen Sie dazu entweder den Quelltext oben oder fügen Sie unterhalb dieses Arbeitsauftrages eine neue Zelle hinzu.
<br><br>
<p style="background-color: yellow; text-align:center">&nbsp</p>

## 4) Daten vorbereiten
Damit die Daten zum Training geeignet sind, müssen sie vorbereitet werden:
<ol>
<li>Sie müssen normalisiert werden.
<li>Der Farbkanal muss hinzugefügt werden.
<li>Die Label müssen in Vektoren umgewandelt werden.
</ol>

1) Für das Training müssen die Daten normalisiert werden. Überlegen Sie, wie Sie Grauwerte zwischen 0 und 255 so normalisieren können, dass sie aus Zahlen zwischen 0 und 1 bestehen.
Ergänzen Sie anschließend den folgenden Quelltext:

In [ ]:
train_data = train_data.astype("float32") / #Ergänzung
test_data = test_data.astype("float32") / #Ergänzung

<details>
<summary>Lösung</summary>
<p>Die Daten müssen durch 255 geteilt werden.</p>
</details>

Zur Überprüfung wird ein normalisiertes Bild ausgegeben.

In [ ]:
print(f"Das Label des Beispielbildes ist: {train_labels[0]}")
train_labels[0]
#Grauwerte als Matrix ausgeben
for row in train_data[0]:
    print(" ".join(f"{pixel:3.2f}" for pixel in row))  # Formatierte Ausgabe ohne zusätzliche Zeilenumbrüche

2) Da ein Bild aus mehreren Farben bestehen könnte, muss beim Training die Anzahl der Farbkanäle mitgegeben werden. In diesem Beispiel gibt es nur einen Farbkanal, da es sich um Graustufen handelt. Dennoch müssen die Daten um diesen Kanal "erweitert" werden.

In [ ]:
train_data = np.expand_dims(train_data, -1)
test_data = np.expand_dims(test_data, -1)

3) Für das Training müssen die **Label** in Vektoren umgewandelt werden. Stellen Sie in Zusammenhang mit der Ausgabe des Netzes eine Vermutung darüber auf, warum diese Umwandlung notwendig ist. <br>
Aktuell sind sie einfach nur Zahlen, z. B. das erste Label:

In [ ]:
print(train_labels[0])

Mit der keras-Funktion utils.to_categorical werden sie in Vektoren umgewandelt, die nur an einer Stelle eine 1 und sonst lauter 0er enthalten.
Erklären Sie, an welcher Stelle die Vektoren die Zahl 1 enthalten.

In [ ]:
num_classes = 10
train_labels = tf.keras.utils.to_categorical(train_labels, num_classes)
test_labels = tf.keras.utils.to_categorical(test_labels, num_classes)

In [ ]:
print(train_labels[0])

## 5) Neuronales Netz (Modell) definieren
Jetzt wird das eigentliche Modell definiert.

In [ ]:
#Es wird ein sequenzielles Modell definiert, in dem die Schichten nacheinander angeordnet sind.
model = tf.keras.Sequential()

#Das Input-Format des Netzes wird definiert.
model.add(tf.keras.layers.Input(shape=(28,28,1)))

#Das Netz wird aufgebaut, indem die Schichten hinzugefügt werden.
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(100, activation='relu'))
model.add(tf.keras.layers.Dense(10, activation='softmax'))

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 6</b></p>
<ol>
   <li style="list-style-type: lower-alpha;">Erklären Sie die drei Zahlen bei shape. Überlegen Sie sich dazu, welches Format die Trainingsdaten haben.</li>
    <li style="list-style-type: lower-alpha;">Geben Sie die Anzahl der Schichten des neuronalen Netzes an.</li>
    <li style="list-style-type: lower-alpha;">Geben Sie die Aktivierungsfunktionen für die letzten beiden Schichten an. Geben Sie zudem den Wertebereich der Funktionen an (recherchieren Sie dazu ggf.).
    <li style="list-style-type: lower-alpha;">Erklären Sie, warum in der letzten Schicht die softmax-Funktion und nicht ReLU verwendet wird. Beschreiben Sie dazu, wie die Ausgabe der 10 Neuronen interpretiert wird.</li>
    <li style="list-style-type: lower-alpha;">Die erste Schicht (layers.Flatten) spielt beim eigentlichen Training keine Rolle. Sie dient dazu, die mehrdimensionalen Eingabedaten in einen eindimensionalen Vektor umzuwandeln. Erklären Sie, warum dieser Schritt notwendig ist und berechnen Sie, wie viele Einträge der Vektor hat.</li>
    
</ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>

Man kann sich das Netz etwas übersichtlicher anzeigen lassen:

In [ ]:
model.summary()

<p style= "background-color: yellow;text-align:center"><b>ARBEITSAUFTRAG 7</b></p>

Vollziehen Sie die Anzahl der Parameter jeder Schicht nach. Überlegen Sie sich dazu jeweils die Eingänge jeder Schicht, die Anzahl der Neuronen sowie die Schwellenwerte. (Hinweis: Dense-Layers sind vollständig verbundene Schichten.)
<br><br>
<p style="background-color: yellow; text-align:center">&nbsp</p>

## 6) Modell trainieren
Jetzt kann das Modell trainiert werden.

In [ ]:
#Modell fürs Training vorbereiten: Art der Kostenfunktion, des Optimizers und der angezeigten Gütekriterien festlegen
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

#Modell trainieren, batch_size und Anzahl der Trainingsepochen festlegen.
model.fit(train_data, train_labels, batch_size=100, epochs=10)

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 8</b></p>
<ol>
   <li style="list-style-type: lower-alpha;">Erklären Sie, warum in einer Epoche 600-mal Anpassungen der Parameter stattfinden. Überlegen Sie sich dazu einen Zusammenhang zwischen der Anzahl der Trainingsdaten und der batch_size.</li>
    <li style="list-style-type: lower-alpha;">Beschreiben Sie die Bedeutung der Werte accuracy sowie loss und analysieren Sie die Werte im Verlauf des Trainings.</li>
    <li style="list-style-type: lower-alpha;">Experimentieren Sie mit verschiedenen Werten für batch_size sowie epochs und vergleichen Sie die Ergebnisse. <strong>Führen Sie bei Veränderungen immer erst erneut den Quelltext zur Definition (!) des Netzes aus, bevor Sie trainieren!</strong></li>
    <li style="list-style-type: lower-alpha;">Verändern Sie auch das Netz (z. B. mehr Neuronen in der Zwischenschicht oder mehr Zwischenschichten) und vergleichen Sie das Ergebnis.</li>
</ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>
## Definition


## 7) Modell mit Testdaten testen
Zum Abschluss wird das Modell mit den Testdaten getestet.

In [ ]:
score = model.evaluate(test_data, test_labels)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

<p style= "background-color: yellow;text-align:center"><b>ARBEITSAUFTRAG 9</b></p>

Ganz am Anfang der Einheit haben Sie einen Wunsch für die Genauigkeit des Systems in unterschiedlichen Kontexten angegeben. Vergleichen Sie Ihre Einschätzung mit der aktuellen Genauigkeit des vorliegenden Systems. Recherchieren Sie ggf., welche Genauigkeit aktuelle KI-Systeme bei der Handschriftenerkennung erreichen.
<br><br>
<p style="background-color: yellow; text-align:center">&nbsp</p>

## 8) Modell verwenden
Mit dem folgenden Quelltext werden drei Vollkommen neu gezeichnete Bilder mit dem Modell klassifiziert.

In [ ]:

images = [Image.open("test2.jpg"), Image.open("test3.jpg")]


# Bilder auf 28x28 Pixel skalieren
images_arrays = []
for image in images:
    image = image.convert('L')  # In Graustufen konvertieren
    image = image.resize((28, 28))  # Auf 28x28 Pixel skalieren

    # Bild in ein NumPy-Array umwandeln
    image_array = np.array(image)

    # Normalisieren der Pixelwerte
    image_array = image_array / 255.0

    # Bild in das Format (28, 28, 1) umwandeln
    image_array = image_array.reshape(28, 28, 1)

    # In Liste speichern
    images_arrays.append(image_array)

# Bild in ein Format für das Modell bringen (Batchgröße von n)
images_arrays = np.array(images_arrays)

In [ ]:
predictions = model.predict(images_arrays)

# Visualisierung der Bilder mit Vorhersagen
fig, axes = plt.subplots(1, len(images), figsize=(10, 3))

for i, ax in enumerate(axes):
    ax.imshow(images_arrays[i].reshape(28, 28), cmap='gray')
    predicted_class = np.argmax(predictions[i])
    ax.set_title(f'Netz sagt vorher: {predicted_class}')
    ax.axis('off')  # Achsen ausschalten

plt.tight_layout()
plt.show()

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 10</b></p>
<ol>
    <li style="list-style-type: lower-alpha;">Stellen Sie in Zusammenhang mit den Trainingsdaten eine Vermutung darüber auf, warum die Ziffer 1 als 7 klassifiziert wird.</li>
    <li style="list-style-type: lower-alpha;">Für Schnelle/Interessierte: Zeichnen Sie ein eigenes Bild und lassen Sie es vom Netz klassifizieren.</li>
  </ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>

## 9) Ausblick
In der Praxis würde ein neuronales Netz zur Erkennung von handschriftlichen Ziffern deutlich komplexer aussehen. Bei der Bilderkennung werden üblicherweise sogenannte Convolutional-Netzwerke eingesetzt. Diese bestehen aus Convolutional-Schichten, die lernen, Muster, Kanten und Formen zu erkennen, und Pooling-Schichten, welche die Daten reduzieren, indem sie überflüssige Informationen entfernen. Identifizieren Sie diese beiden Schichtarten in der Definition des Netztes im Quelltext unten.
Trainieren Sie das Netz und vergleichen Sie die Genauigkeit mit der des einfachen Netzes.
Wenden Sie es ggf. auf neue Testbilder an.

In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ]
)

In [ ]:
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

model.fit(train_data, train_labels, batch_size=100, epochs=10,validation_split=0.1)



In [ ]:
score = model.evaluate(test_data, test_labels)#, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])